# ✅ Auth + Setup

## 🔐 **Authenticate to Google Cloud within Colab**

Authenticate to Google Cloud as the IAM user logged into this notebook in order to access your Google Cloud Project.

In [ ]:
from google.colab import auth

auth.authenticate_user()

## 💻 **Install Code Dependencies**
It is recommended to use the Connector alongside a library that can create connection pools, such as [SQLAlchemy](https://www.sqlalchemy.org/).
This will allow for connections to remain open and be reused, reducing connection overhead and the number of connections needed

Let's `pip install` the [Cloud SQL Python Connector](https://github.com/GoogleCloudPlatform/cloud-sql-python-connector) as well as [SQLAlchemy](https://www.sqlalchemy.org/), using the below command.

In [ ]:
# install dependencies
import sys
!{sys.executable} -m pip install cloud-sql-python-connector["pymysql"] SQLAlchemy==2.0.7

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 55.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 77.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.5/114.5 kB 14.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 33.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.6/149.6 kB 17.5 MB/s eta 0:00:00
  Attempting uninstall: SQLAlchemy
    Found existing installation: SQLAlchemy 2.0.10
    Uninstalling SQLAlchemy-2.0.10:
      Successfully uninstalled SQLAlchemy-2.0.10


In [ ]:
import google.auth
import pandas as pd

from google.cloud.sql.connector import Connector
from google.auth.transport.requests import Request
from sqlalchemy import create_engine, Table, Column, Integer, VARCHAR, ForeignKey, String, MetaData, text, select
from sqlalchemy.orm import Session

## 🐬 **Connect to a MySQL Instance**
We are now ready to connect to a MySQL instance using the Cloud SQL Python Connector! 🐍 ⭐ ☁


In [ ]:
# initialize parameters
INSTANCE_CONNECTION_NAME = "inlaid-woods-388716:us-west1:ilkmaar" # i.e demo-project:us-central1:demo-instance
print(f"Your instance connection name is: {INSTANCE_CONNECTION_NAME}")

# IAM database user parameter (IAM user's email before the "@" sign, mysql truncates usernames)
# ex. IAM user with email "demo-user@test.com" would have database username "demo-user"
# grant Cloud SQL Client role to authenticated user
current_user = !gcloud auth list --filter=status:ACTIVE --format="value(account)"

IAM_USER = current_user[0].split("@")[0]
DB_NAME = "gameplay-data"

Your instance connection name is: inlaid-woods-388716:us-west1:ilkmaar


### ✅ **Connect to Database**
To connect to Cloud SQL using the connector, initialize a `Connector` object and call its `connect` method with the proper input parameters.

In [ ]:
# initialize connector
connector = Connector()

# getconn now using IAM user and requiring no password with IAM Auth enabled
def getconn():
    conn = connector.connect(
      INSTANCE_CONNECTION_NAME,
      "pymysql",
      user=IAM_USER,
      db=DB_NAME,
      enable_iam_auth=True
    )
    return conn

# create connection pool
engine = create_engine(
    "mysql+pymysql://",
    creator=getconn,
)

def query_db(query_str):
    with engine.connect() as conn:
        return pd.read_sql_query(text(query_str), conn)

def describe(table_name):
    with engine.connect() as conn:
        res = conn.execute(text(f"DESCRIBE {table_name};"))
        df = pd.DataFrame(res.fetchall(), columns=res.keys())
        return df

def drop_if_exists(table_name):
    with engine.connect() as conn:
        res = conn.execute(text(f"DROP TABLE IF EXISTS {table_name};"))
        return res

### Test Connection

This fails if the user does not have access to that database yet.

To fix:
  - log into the MySQL server on the Google Cloud Shell as root user
  - MySQL > GRANT ALL PRIVILEGES on `gameplay-data`.* to "user"@'%'

In [ ]:
# connect to connection pool
with engine.connect() as db_conn:
    # get current datetime from database
    results = db_conn.execute(text("SELECT NOW()")).fetchone()

    # output time
    print("Current time: ", results[0])

Current time:  2023-06-08 18:13:48


# 📈 Load Data from Google Sheets

## **Read in 'World Setup Data' and 'Junctions' Google Sheets**

In [ ]:
import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

world_data = gc.open('World Setup Data')
junctions_data = gc.open('Junctions')

base_tables = ['creatures', 'resource_types', 'location_groups', 'recipes', 'players', 'collections']
secondary_tables = ['recipe_ingredient_resource_types', 'locations']

worksheets = {}
for table_name in base_tables:
    worksheets[table_name] = world_data.worksheet(table_name)

for table_name in secondary_tables:
    worksheets[table_name] = junctions_data.worksheet(table_name)

## **Create dataframes from 'World Setup Data' worksheets**

In [ ]:
def sanitize_column_name(table_name):
    if table_name.endswith('s'):
        table_name = table_name[:-1]
    return table_name + '_id'

def drop_table_if_exists(table_name):
    if(table_name in metadata.tables):
        table = metadata.tables[table_name]
        table.drop(engine, checkfirst=True)
    else:
        drop_if_exists(table_name)

def create_SQLA_table(table_name, df):
    # Define the columns.
    columns = [Column(sanitize_column_name(table_name), Integer, primary_key=True, autoincrement=True)]
    for column_name in df.columns:
        if(column_name in ['creature_health', 'creature_mood', 'collection_id', 'resource_type_rarity', 'player_level', 'recipe_temp', 'recipe_difficulty', 'resource_type_base_mood_effect', 'resource_type_base_health_effect', 'recipe_base_mood_effect', 'recipe_base_health_effect']):
            columns.append(Column(column_name, Integer))
        else:
            columns.append(Column(column_name, String(255)))

    # Define the table
    table = Table(table_name, metadata, *columns, extend_existing=True)
    return table

In [ ]:
def sanitize_table_name(name):
    return "".join(c if c.isalnum() else "_" for c in name)

def create_dataframe(table_name):
    # Load the worksheet into a pandas DataFrame
    df = pd.DataFrame(worksheets[table_name].get_all_records())

    # Sanitize column names to replace spaces with underscores
    df.columns = df.columns.str.replace(' ', '_')

    return df

data_frames = {}

# Iterate over all worksheets in the Google Sheet
for table_name in base_tables:
    df = create_dataframe(table_name)
    data_frames[table_name] = df

# ❌ Drop Tables if Exist

In [ ]:
metadata = MetaData()

In [ ]:
for table_name in ['interaction_events']:
  drop_table_if_exists(table_name)

for table_name in ['crafting_events', 'gifting_events']:
  drop_table_if_exists(table_name)

for table_name in ['resource_transfers', 'item_transfers']:
  drop_table_if_exists(table_name)

for table_name in ['player_creature_relationships', 'player_collection_access', 'recipe_ingredient_resource_types', 'resources', 'items', 'locations']:
  drop_table_if_exists(table_name)

for table_name in base_tables:
  drop_table_if_exists(table_name)

In [ ]:
query_db("SHOW TABLES;")

,Tables_in_gameplay-data


# 📓 Define Tables

### **location_groups, players, creatures, resource_types, recipes, collections**

In [ ]:
# Define all base tables: ['creatures', 'resource_types', 'location_groups', 'recipes', 'players', 'collections']
for table_name in base_tables:
    df = data_frames[table_name]
    table = create_SQLA_table(table_name, df)

# Create all tables from metadata
metadata.create_all(engine)

In [ ]:
# Create a new column named 'collection_id' with a foreign key to the 'collections' table
with engine.connect() as connection:
    connection.execute(text('ALTER TABLE creatures ADD COLUMN collection_id INTEGER;'))
    connection.execute(text('ALTER TABLE creatures ADD CONSTRAINT fk_collection_id FOREIGN KEY(collection_id) REFERENCES collections(collection_id);'))

metadata.reflect(engine)

### **locations**
*uses: location_groups*

*used by: interaction_events*

In [ ]:
location_groups = metadata.tables['location_groups']

# Define the table
locations = Table(
    'locations', metadata,
    Column('location_id', Integer, primary_key=True, autoincrement=True),
    Column('location_x', Integer),
    Column('location_y', Integer),
    Column('location_group_id', Integer, ForeignKey('location_groups.location_group_id')),
    extend_existing=True
)

### **player_creature_relationships**
*uses: players, creatures*

In [ ]:
players = metadata.tables['players']
creatures = metadata.tables['creatures']

# Define the 'Relationships' columns.
relationships_columns = [
    Column('relationship_id', Integer, primary_key=True, autoincrement=True),
    Column('player_id', Integer, ForeignKey('players.player_id')),
    Column('creature_id', Integer, ForeignKey('creatures.creature_id')),
    Column('player_creature_relationship_level', Integer)
]

# Define the 'Relationships' table
player_creature_relationships = Table('player_creature_relationships', metadata, *relationships_columns, extend_existing=True)

### **player_collection_access**
*uses: players, collections*

In [ ]:
players = metadata.tables['players']
collections = metadata.tables['collections']

# Define the 'CollectionAccess' columns.
player_collection_access_columns = [
    Column('player_collection_access_id', Integer, primary_key=True, autoincrement=True),
    Column('player_id', Integer, ForeignKey('players.player_id')),
    Column('collection_id', Integer, ForeignKey('collections.collection_id')),
    Column('player_collection_access_level', Integer)
]

# Define the 'CollectionsAccess' table
player_collection_access = Table('player_collection_access', metadata, *player_collection_access_columns, extend_existing=True)

### **recipe_ingredient_resource_types**
*uses: recipes, resource_types*

In [ ]:
recipes = metadata.tables['recipes']
resource_types = metadata.tables['resource_types']

recipe_ingredient_resource_types = Table(
    'recipe_ingredient_resource_types', metadata,
    Column('recipe_ingredient_resource_type_id', Integer, primary_key=True, autoincrement=True),
    Column('recipe_id', Integer, ForeignKey('recipes.recipe_id')),
    Column('resource_type_id', Integer, ForeignKey('resource_types.resource_type_id')),
    extend_existing=True
)

### **resources** and **items**

*uses: resource_types, recipes, collections*;

*used by: resource_transfers, item_transfers*

In [ ]:
resource_types = metadata.tables['resource_types']
collections = metadata.tables['collections']
recipes = metadata.tables['recipes']

resources = Table(
   'resources', metadata,
   Column('resource_id', Integer, primary_key=True, autoincrement=True),
   Column('resource_quality', Integer),
   Column('resource_type_id', Integer, ForeignKey('resource_types.resource_type_id')),
   Column('collection_id', Integer, ForeignKey('collections.collection_id')),
   extend_existing=True
)

items = Table(
   'items', metadata,
   Column('item_id', Integer, primary_key=True, autoincrement=True),
   Column('item_quality', Integer),
   Column('recipe_id', Integer, ForeignKey('recipes.recipe_id')),
   Column('collection_id', Integer, ForeignKey('collections.collection_id')),
   extend_existing=True
)

### **resource_transfers** and **item_transfers**
*uses: collections, resources, items*

In [ ]:
collections = metadata.tables['collections']
resources = metadata.tables['resources']
items = metadata.tables['items']

resource_transfers = Table(
    'resource_transfers', metadata,
    Column('resource_transfer_id', Integer, primary_key=True, autoincrement=True),
    Column('resource_transfer_time', Integer),
    Column('source_collection_id', Integer, ForeignKey('collections.collection_id')),
    Column('destination_collection_id', Integer, ForeignKey('collections.collection_id')),
    Column('resource_id', Integer, ForeignKey('resources.resource_id')),
    extend_existing=True
)

item_transfers = Table(
    'item_transfers', metadata,
    Column('item_transfer_id', Integer, primary_key=True, autoincrement=True),
    Column('item_transfer_time', Integer),
    Column('source_collection_id', Integer, ForeignKey('collections.collection_id')),
    Column('destination_collection_id', Integer, ForeignKey('collections.collection_id')),
    Column('item_id', Integer, ForeignKey('items.item_id')),
    extend_existing=True
)

### **crafting_events**
*uses: recipes, item_transfers, collections*

In [ ]:
recipes = metadata.tables['recipes']
item_transfers = metadata.tables['item_transfers']
collections = metadata.tables['collections']

# Crafting Events
crafting_events = Table(
    'crafting_events', metadata,
    Column('crafting_event_id', Integer, primary_key=True),
    Column('crafting_event_time', Integer),
    Column('crafting_event_skill_level', Integer),
    Column('recipe_id', Integer, ForeignKey('recipes.recipe_id')),
    Column('item_transfer_id', Integer, ForeignKey('item_transfers.item_transfer_id')),
    Column('crafting_table_collection_id', Integer, ForeignKey('collections.collection_id')),
    extend_existing=True
)

#CraftingInputEvents = Table(
#    'CraftingInputEvents', metadata,
#    Column('transfer_event_id', Integer, ForeignKey('ResourceTransferEvents.id'), primary_key=True),
#    Column('crafting_event_id', Integer, ForeignKey('CraftingOutputEvents.id'), primary_key=True),
#    extend_existing=True
#)

### **gifting_events**
*uses: players, creatures, item_transfers*

*used by: interaction_events*

In [ ]:
players = metadata.tables['players']
creatures = metadata.tables['creatures']
item_transfers = metadata.tables['item_transfers']

# Gifting Events
gifting_events = Table(
    'gifting_events', metadata,
    Column('gifting_event_id', Integer, primary_key=True),
    Column('gifting_event_time', Integer),
    Column('player_id', Integer, ForeignKey('players.player_id')),
    Column('creature_id', Integer, ForeignKey('creatures.creature_id')),
    Column('item_transfer_id', Integer, ForeignKey('item_transfers.item_transfer_id')),
    extend_existing=True
)

### **interaction_events**
*uses: players, creatures, locations, gifting_events*

In [ ]:
players = metadata.tables['players']
creatures = metadata.tables['creatures']
locations = metadata.tables['locations']
gifting_events = metadata.tables['gifting_events']

# Interaction Events
interaction_events = Table(
    'interaction_events', metadata,
    Column('interaction_event_id', Integer, primary_key=True),
    Column('interaction_event_time', Integer),
    Column('player_id', Integer, ForeignKey('players.player_id')),
    Column('creature_id', Integer, ForeignKey('creatures.creature_id')),
    Column('interaction_event_observed_creature_mood', Integer),
    Column('location_id', Integer, ForeignKey('locations.location_id')),
    Column('gifting_event_id', Integer, ForeignKey('gifting_events.gifting_event_id')),
    extend_existing=True
)

### **creature_sightings** (represents a "watcher")
*uses: time, creatures, location_groups*

In [ ]:
creatures = metadata.tables['creatures']
location_groups = metadata.tables['location_groups']

# Interaction Events
creature_sightings = Table(
    'creature_sightings', metadata,
    Column('creature_sighting_id', Integer, primary_key=True),
    Column('creature_sighting_time', Integer),
    Column('creature_id', Integer, ForeignKey('creatures.creature_id')),
    Column('location_group_id', Integer, ForeignKey('location_groups.location_group_id')),
    extend_existing=True
)

In [ ]:
metadata.create_all(engine)

# 🪣 Populate Tables

### Populate base tables from dataframes

In [ ]:
for table_name in base_tables:
    # Insert the data from the DataFrame
    df = data_frames[table_name];
    df.to_sql(table_name, engine, if_exists='append', index=False)

### Populate locations

In [ ]:
locations = metadata.tables['locations']

# get MapLocations data
df = create_dataframe('locations')

with engine.begin() as conn:
  for index, row in df.iterrows():
      # Get the ID of the Map Area
      location_group_id = conn.execute(text(
          f"SELECT location_group_id FROM location_groups WHERE location_group_name = '{row['location_group']}'"
      )).scalar()

      # Create a new entry in the 'RecipeIngredients' table
      conn.execute(
          locations.insert().values(location_group_id=location_group_id, location_x=row['location_x'], location_y=row['location_y'])
      )

### Populate recipe_ingredient_resource_types

In [ ]:
recipe_ingredient_resource_types = metadata.tables['recipe_ingredient_resource_types']

# get recipe_ingredient_resource_types data
df = create_dataframe('recipe_ingredient_resource_types')

with engine.begin() as conn:
  for index, row in df.iterrows():
      # Get the recipe ID
      recipe_id = conn.execute(text(
          f"SELECT recipe_id FROM recipes WHERE recipe_name = '{row['recipe']}'"
      )).scalar()

      # For each ingredient in the 'ingredients' list
      ingredient_names = row['resource_types'].split(', ')

      for ingredient_name in ingredient_names:
          # Get the resource ID
          resource_type_id = conn.execute(text(
              f"SELECT resource_type_id FROM resource_types WHERE resource_type = '{ingredient_name}'"
          )).scalar()

          # Create a new entry in the 'recipe_resources' table
          conn.execute(
              recipe_ingredient_resource_types.insert().values(recipe_id=recipe_id, resource_type_id=resource_type_id)
          )

### Populate player_creature_relationships table

In [ ]:
player_creature_relationships = metadata.tables['player_creature_relationships']

creatures = query_db("SELECT creature_id, creature_name FROM creatures")
players = query_db("SELECT player_id, player_name FROM players")

with engine.begin() as conn:
    for i, player in players.iterrows():
        # Get the player ID
        player_id = player['player_id']

        for i, creature in creatures.iterrows():
            creature_id = creature['creature_id']

            # Create a new entry in the 'Relationships' table
            conn.execute(
                player_creature_relationships.insert().values(player_id=player_id, creature_id=creature_id, player_creature_relationship_level=1)
            )

### Populate collections + player_collection_access tables

In [ ]:
collections = metadata.tables['collections']

#### Create a collection for each map area (location group)

In [ ]:
def populate_collections_from_location_groups():
    location_groups_df = data_frames['location_groups']

    # Filter the location_groups that are of type 'collection_area'
    location_groups_for_collection = location_groups_df[location_groups_df['location_group_type'] == 'collection_area']

    # Prepare the data for insertion into the Collections table
    collections_data = pd.DataFrame()
    collections_data['collection_name'] = location_groups_for_collection['location_group_name']
    collections_data['collection_type'] = 'map_area'

    # Insert the data into the Collections table
    # with engine.begin() as conn:
    collections_data.to_sql('collections', engine, if_exists='append', index=False)

populate_collections_from_location_groups()

#### Create a collection for each creature

In [ ]:
# Load the table object for creatures
creatures = Table('creatures', metadata, autoload_with=engine)
collections = Table('collections', metadata, autoload_with=engine)

creatures_df = query_db("SELECT creature_id, creature_name from creatures")

for _, creature in creatures_df.iterrows():
    creature_id = creature['creature_id']
    creature_name = creature['creature_name']

    with engine.begin() as conn:
        # Insert new collection to 'Collections' table
        collection_insert = collections.insert().values({"collection_name": f"{creature_name}'s Inventory", "collection_type": "creature"})
        result = conn.execute(collection_insert)

        # Get the collection ID of the newly inserted collection
        collection_id = result.inserted_primary_key[0]

        # Update the creature's collection_id
        conn.execute(text(f"""UPDATE creatures
        SET collection_id = {collection_id}
        WHERE creature_id = {creature_id};
        """))

#### Create three collections for each player: inventory, crafting table, and shop

In [ ]:
player_collection_access = metadata.tables['player_collection_access']

players_df = query_db("SELECT player_id, player_name FROM players")

for _, player in players_df.iterrows():
    player_id = player['player_id']
    player_name = player['player_name']

    # Create new collections
    collections_list = [
        {"collection_name": f"{player_name}'s Inventory", "collection_type": "inventory"},
        {"collection_name": f"{player_name}'s Crafting Table", "collection_type": "crafting_table"},
        {"collection_name": f"{player_name}'s Shop", "collection_type": "shop"},
    ]

    with engine.begin() as conn:
        for collection in collections_list:
            # Insert new collection to 'Collections' table
            ins = collections.insert().values(collection)
            result = conn.execute(ins)

            # Get the collection ID of the newly inserted collection
            collection_id = result.inserted_primary_key[0]

            # Create CollectionAccess entry
            collection_access = {
                "player_id": player_id,
                "collection_id": collection_id,
                "player_collection_access_level": 1,
            }

            # Insert to 'CollectionAccess' table
            collection_access_insert = player_collection_access.insert().values(collection_access)
            conn.execute(collection_access_insert)